# <center> **Proyecto visión por computador:** Attention Residual U-Net </center>

**Por:** María José Bernal, Alejandro Olivares, Santiago Zapata y Jerly Alejandra Galeano

Este notebook entrena y evalúa el modelo **Attention Residual U-Net** con el protocolo común del proyecto, para que su resultado sea comparable con el de los demás modelos.

**Qué pone a prueba.** ¿Aportan los puentes con compuertas de atención y los bloques residuales? Tiene casi los mismos parámetros que la línea base y también se entrena desde cero, así que la comparación con la CNN aísla el efecto de la arquitectura.

| Elemento | Valor |
|---|---|
| Tamaño de entrada | 350x525 con relleno negro hasta 352x544 |
| Partición | 80/10/10 estratificada (`splits/split.csv`), 4438 / 553 / 555 imágenes |
| Salida | 4 máscaras, una por clase, con sigmoide independiente |
| Pérdida | Focal (γ = 2) + Dice |
| Optimización | AdamW + OneCycleLR, 12 épocas, lote de 16, precisión mixta |
| Selección de la época | Mejor Dice global en validación, con umbral 0.5 |
| Posprocesado | Umbral, área mínima y morfología, calibrados en validación |
| Evaluación | Una sola vez en prueba, sobre 350x525 sin el relleno |

## 1. Configuración
`ENTRENAR = True` entrena el modelo desde cero. Con `ENTRENAR = False` el notebook no entrena y usa los pesos ya guardados en `artifacts/`, para repetir la evaluación o las gráficas sin volver a entrenar.

In [ ]:
import sys
import time
import warnings
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")
%config InlineBackend.figure_formats = ["jpeg"]
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from config import IN_H, IN_W, LABELS, MEAN, METRICA_PRINCIPAL, PROTOCOLO, STD, carpeta_modelo
from datos import CloudDataset, preparar_datos
from entrenamiento import (FocalDice, cargar_pesos, crear_loaders, crear_optimizacion, dispositivo,
                           estado_cpu, fijar_semilla, guardar_pesos, validar)
from evaluacion import evaluar_variantes, guardar_resultados, predecir, tiempo_cpu_ms
from metricas import dice_por_imagen, tabla
from posproceso import aplicar, calibrar
import graficas

MODELO = "att_resunet"
CARPETA = carpeta_modelo(MODELO)
ENTRENAR = True

fijar_semilla()
device = dispositivo()
print(f"Dispositivo: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))
print(f"Métrica principal: {METRICA_PRINCIPAL}")
print(f"Protocolo: {PROTOCOLO}")
print(f"Artefactos en: {CARPETA}")

## 2. Datos
La preparación reduce las 5546 imágenes a 350x525, les agrega el relleno negro hasta 352x544, convierte cada RLE en una máscara de 4 canales y asigna cada imagen a su partición según `splits/split.csv`. Se hace una sola vez por sesión. Si los datos ya están preparados, solo se leen.

In [ ]:
indice = preparar_datos()
train_dl, val_dl = crear_loaders()
print(f"Lotes por época: {len(train_dl)} de entrenamiento y {len(val_dl)} de validación")

### 2.1. Ejemplos de entrenamiento con aumento de datos
Flips, transformación afín suave y cambios leves de brillo y contraste, aplicados igual a la imagen y a sus máscaras.

In [ ]:
x, y = next(iter(train_dl))
fig, axes = plt.subplots(1, 4, figsize=(22, 3.8))
for ax, img, mask in zip(axes, x[:4], y[:4]):
    img = (np.clip(img.numpy().transpose(1, 2, 0) * STD + MEAN, 0, 1) * 255).astype(np.uint8)
    ax.imshow(graficas.superponer(img, mask.numpy() > 0.5))
    ax.axis("off")
graficas.leyenda(fig)
fig.suptitle("Ejemplos de entrenamiento después del aumento", fontweight="bold")
plt.show()

## 3. Implementación del modelo
U-Net entrenada desde cero con dos mejoras sobre la versión clásica (8.2 M de parámetros).

- **Bloques residuales.** Cada bloque suma su entrada a su salida a través de un atajo, lo que facilita que el gradiente fluya y estabiliza el entrenamiento.
- **Puentes (skip connections).** Cada nivel del encoder envía su mapa de características al nivel correspondiente del decoder, que así recupera el detalle espacial perdido al reducir la resolución.
- **Compuertas de atención (Oktay et al., 2018).** Antes de concatenar cada puente, la señal del decoder calcula un mapa entre 0 y 1 que lo pondera. Así se resaltan las regiones relevantes y se atenúa el resto.
- **Salida.** Una convolución 1x1 con 4 canales, uno por clase.

La celda siguiente define el modelo y lo guarda en `src/arquitecturas/att_resunet.py`, para que el notebook del campeón y la app usen exactamente la misma implementación.

In [ ]:
%%writefile {ROOT}/src/arquitecturas/att_resunet.py
import torch
import torch.nn as nn


class ResBlock(nn.Module):
    """Bloque residual: dos convoluciones 3x3 más un atajo que suma la entrada a la salida."""

    def __init__(self, c_in, c_out):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out),
        )
        self.atajo = nn.Identity() if c_in == c_out else nn.Sequential(
            nn.Conv2d(c_in, c_out, 1, bias=False), nn.BatchNorm2d(c_out))
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.body(x) + self.atajo(x))


class AttentionGate(nn.Module):
    """Compuerta de atención (Oktay et al., 2018): la señal del decoder pondera el puente entre 0 y 1."""

    def __init__(self, c_gate, c_skip, c_int):
        super().__init__()
        self.w_g = nn.Sequential(nn.Conv2d(c_gate, c_int, 1, bias=False), nn.BatchNorm2d(c_int))
        self.w_x = nn.Sequential(nn.Conv2d(c_skip, c_int, 1, bias=False), nn.BatchNorm2d(c_int))
        self.psi = nn.Sequential(nn.Conv2d(c_int, 1, 1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, gate, skip):
        return skip * self.psi(self.relu(self.w_g(gate) + self.w_x(skip)))


class AttResUNet(nn.Module):
    """U-Net con bloques residuales y compuertas de atención en los puentes, entrenada desde cero."""

    def __init__(self, classes=4, chans=(32, 64, 128, 256), bottleneck=512):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.enc = nn.ModuleList()
        c_in = 3
        for c in chans:
            self.enc.append(ResBlock(c_in, c))
            c_in = c
        self.bottleneck = ResBlock(c_in, bottleneck)
        self.up, self.att, self.dec = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        c_in = bottleneck
        for c in reversed(chans):
            self.up.append(nn.ConvTranspose2d(c_in, c, 2, stride=2))
            self.att.append(AttentionGate(c, c, c // 2))
            self.dec.append(ResBlock(2 * c, c))
            c_in = c
        self.head = nn.Conv2d(c_in, classes, 1)

    def forward(self, x):
        puentes = []
        for bloque in self.enc:
            x = bloque(x)
            puentes.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        for up, att, dec, puente in zip(self.up, self.att, self.dec, reversed(puentes)):
            x = up(x)
            x = dec(torch.cat([att(x, puente), x], dim=1))
        return self.head(x)


def construir_modelo(preentrenado=True):
    return AttResUNet()

In [ ]:
import arquitecturas.att_resunet as arquitectura
importlib.reload(arquitectura)

modelo = arquitectura.construir_modelo(preentrenado=False).to(device)
n_params = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
modelo.eval()
with torch.no_grad():
    salida = modelo(torch.zeros(1, 3, 352, 544, device=device))
print(f"Parámetros entrenables: {n_params / 1e6:.1f} M")
print(f"Entrada (1, 3, 352, 544) -> salida {tuple(salida.shape)}")

## 4. Pérdida y optimización
La pérdida es la misma para todos los modelos y suma dos términos.

$$\mathcal{L} = \mathcal{L}_{focal} + \mathcal{L}_{Dice}$$

$$\mathcal{L}_{focal} = -\frac{1}{M}\sum_{m} (1 - p_{t,m})^{\gamma} \log p_{t,m} \qquad \mathcal{L}_{Dice} = 1 - \frac{1}{4}\sum_{k} \frac{2 \sum_m p_{k,m}\, y_{k,m}}{\sum_m p_{k,m} + \sum_m y_{k,m}}$$

- $m$ recorre los $M$ píxeles del lote y $k$ las 4 clases.
- $p_{k,m}$ es la probabilidad que predice el modelo para la clase $k$ en el píxel $m$, e $y_{k,m}$ es la etiqueta (0 o 1).
- $p_{t,m}$ es la probabilidad asignada al valor correcto del píxel, es decir, $p_{k,m}$ si la etiqueta es 1 y $1 - p_{k,m}$ si es 0.
- $\gamma = 2$ reduce el peso de los píxeles fáciles, para que el entrenamiento se concentre en los difíciles.

La focal trabaja píxel a píxel y la Dice mide el solapamiento de cada máscara completa, que es lo que evalúa la métrica. Se optimiza con AdamW y OneCycleLR, que sube la tasa de aprendizaje hasta 1e-3 y luego la baja hasta casi cero al final de las 12 épocas.

In [ ]:
criterio = FocalDice()
opt, sched, scaler = crear_optimizacion(modelo, len(train_dl))

## 5. Entrenamiento
En cada época se recorre todo el conjunto de entrenamiento y luego se mide la pérdida y las métricas en validación, con umbral 0.5 y sin el relleno. Se conserva la época con mejor métrica principal. Al final, sus pesos se guardan en media precisión (fp16), que ocupa la mitad del espacio.

In [ ]:
if ENTRENAR:
    usar_amp = device.type == "cuda"
    historial, mejor, mejor_estado, mejor_epoca = [], -1.0, None, 0
    for epoca in range(1, PROTOCOLO["epocas"] + 1):
        modelo.train()
        t0, suma, n = time.time(), 0.0, 0
        for x, y in train_dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=usar_amp):
                logits = modelo(x)
            perdida = criterio(logits.float(), y)
            scaler.scale(perdida).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), PROTOCOLO["clip_gradiente"])
            scaler.step(opt)
            scaler.update()
            sched.step()
            suma += perdida.item() * len(x)
            n += len(x)

        fila = {"epoca": epoca, "perdida_train": suma / n, **validar(modelo, val_dl, criterio, device)}
        fila["segundos"] = time.time() - t0
        historial.append(fila)
        pd.DataFrame(historial).to_csv(CARPETA / "historial.csv", index=False)
        marca = ""
        if fila[METRICA_PRINCIPAL] > mejor:
            mejor, mejor_epoca, mejor_estado = fila[METRICA_PRINCIPAL], epoca, estado_cpu(modelo)
            marca = "  <- mejor"
        print(f"Época {epoca:2d} | pérdida train {fila['perdida_train']:.4f} | val {fila['perdida_val']:.4f} | "
              f"Dice {fila['dice']:.4f} | IoU {fila['iou']:.4f} | Dice Kaggle {fila['dice_kaggle']:.4f} | "
              f"{fila['segundos']:.0f}s{marca}", flush=True)

    historial = pd.DataFrame(historial)
    guardar_pesos(mejor_estado, MODELO, {"epoca": mejor_epoca, "valor_validacion": mejor,
                                          "metrica": METRICA_PRINCIPAL, "n_params": n_params,
                                          "protocolo": PROTOCOLO})
else:
    historial = pd.read_csv(CARPETA / "historial.csv")
    print("Se omite el entrenamiento y se usan los pesos guardados.")

historial.round(4)

## 6. Curvas de entrenamiento

In [ ]:
fig = graficas.curvas(historial, "Attention Residual U-Net")
fig.savefig(CARPETA / "curvas.png", dpi=110, bbox_inches="tight")
plt.show()

mejor_fila = historial.loc[historial[METRICA_PRINCIPAL].idxmax()]
print(f"Mejor época: {int(mejor_fila['epoca'])} con {METRICA_PRINCIPAL} de validación {mejor_fila[METRICA_PRINCIPAL]:.4f}")

## 7. Calibración del posprocesado en validación
Se recargan los pesos fp16 guardados, para evaluar exactamente el modelo que usará la app. Sobre validación se comparan tres variantes de posprocesado.

- **A. Umbral 0.5.** Un píxel es de la clase si su probabilidad supera 0.5.
- **B. Umbral por clase.** Se elige el umbral de cada clase que maximiza la métrica principal.
- **C. Umbral + área mínima + morfología.** Además del umbral, se prueba un cierre y una apertura morfológica (rellenar huecos y borrar regiones pequeñas) y un área mínima por debajo de la cual la máscara se borra completa.

Los parámetros se eligen solo con validación y se aplican sin cambios en prueba.

In [ ]:
modelo = arquitectura.construir_modelo(preentrenado=False)
meta = cargar_pesos(modelo, MODELO, device)
print(f"Pesos fp16 cargados (época {meta['epoca']})")

probs_val, targets_val, _ = predecir(modelo, "val", device)
config_post = calibrar(probs_val, targets_val)
display(pd.DataFrame(config_post).T)

cnt_val, tabla_val = evaluar_variantes(probs_val, targets_val, config_post)
del probs_val, targets_val
print("Validación (promedio de las 4 clases):")
tabla_val.round(4)

## 8. Evaluación en prueba
Las 555 imágenes de prueba no participaron en ninguna decisión (época, umbral ni posprocesado). Este es el resultado que se reporta.

In [ ]:
probs_test, targets_test, ms_lote = predecir(modelo, "test", device)
cnt_test, tabla_test = evaluar_variantes(probs_test, targets_test, config_post)
print("Prueba (promedio de las 4 clases):")
display(tabla_test.round(4))
print("Prueba por clase con el posprocesado completo (variante C):")
tabla(cnt_test["C"]).round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
por_clase = tabla_test[[f"dice_{l}" for l in LABELS]].T
por_clase.index = LABELS
por_clase.plot.bar(ax=ax, rot=0, color=["#b3cde0", "#6497b1", "#03396c"])
ax.set_ylabel("Dice global en prueba")
ax.set_ylim(0, 1)
ax.set_title("Dice por clase según la variante de posprocesado", fontweight="bold")
ax.grid(axis="y", alpha=0.3)
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(CARPETA / "dice_por_clase.png", dpi=110)
plt.show()

## 9. Visualización de predicciones
Las tres imágenes de prueba con mejor Dice y las tres con peor, con el posprocesado completo. Después, los mapas de probabilidad de una imagen típica (la de Dice mediano), con el contorno de la etiqueta en blanco.

In [ ]:
test_ds = CloudDataset("test")
pred_test = aplicar(probs_test, config_post, "C")
dice_img = dice_por_imagen(cnt_test["C"])
orden = np.argsort(dice_img)
elegidos = list(orden[-3:][::-1]) + list(orden[:3])
imagenes = [test_ds.imagen_y_mascara(i)[0][:IN_H, :IN_W] for i in elegidos]
fig = graficas.ejemplos(imagenes, targets_test[elegidos], pred_test[elegidos], dice_img[elegidos],
                        ["mejor"] * 3 + ["peor"] * 3)
fig.savefig(CARPETA / "ejemplos.jpg", dpi=110, pil_kwargs={"quality": 85})
plt.show()

In [ ]:
i = int(orden[len(orden) // 2])
fig = graficas.mapas_probabilidad(test_ds.imagen_y_mascara(i)[0][:IN_H, :IN_W], probs_test[i], targets_test[i])
fig.savefig(CARPETA / "mapas_probabilidad.jpg", dpi=110, bbox_inches="tight", pil_kwargs={"quality": 85})
plt.show()

## 10. Guardado de artefactos
Todo queda en `artifacts/<modelo>/`, para comparar modelos y desplegar el campeón sin volver a entrenar.

| Archivo | Contenido |
|---|---|
| `pesos_fp16.pt` | Pesos de la mejor época en media precisión |
| `historial.csv` | Pérdida y métricas por época |
| `posproceso.json` | Umbral, área mínima y morfología por clase |
| `metricas.csv` | Todas las métricas por partición, variante y clase |
| `conteos.npz` | Intersección y áreas por imagen y clase, para el bootstrap del notebook del campeón |
| `resumen.json` | Cifras principales del modelo |
| `resultados.pkl` | Todo lo anterior (salvo los pesos) en un solo objeto de Python |
| `*.png`, `*.jpg` | Curvas, Dice por clase, ejemplos y mapas de probabilidad |

In [ ]:
info = {
    "n_params": n_params,
    "mejor_epoca": int(mejor_fila["epoca"]),
    "ms_por_imagen_evaluacion": round(ms_lote, 2),
    "dispositivo_evaluacion": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
    "ms_por_imagen_cpu": round(tiempo_cpu_ms(modelo), 1),
}
resumen = guardar_resultados(MODELO, historial, config_post, cnt_val, cnt_test, tabla_val, tabla_test, info)
resumen

## 11. Resumen

In [ ]:
print(f"{resumen['nombre']} ({n_params / 1e6:.1f} M parámetros, mejor época {resumen['mejor_epoca']})")
for v, nombre in [("A", "umbral 0.5"), ("B", "umbral por clase"), ("C", "posprocesado completo")]:
    print(f"  {METRICA_PRINCIPAL} con {nombre:22s} validación {resumen['validacion'][v]:.4f} | prueba {resumen['prueba'][v]:.4f}")
print(f"  Inferencia en CPU: {resumen['ms_por_imagen_cpu']:.0f} ms por imagen")
print("La comparación entre modelos y la elección del campeón están en 05_champion.ipynb")